# Protein Aggregation, Solubility, and Structural Measurements for Escherichia coli Cytoplasmic Proteins under Macromolecular Crowding Conditions Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.


In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.k96t-6h6x/fair2.json"

# Load the dataset metadata and create a Dataset instance
dataset = mlc.Dataset(croissant_url)
# The metadata attribute is an object, use .name and .description
print(f"{dataset.metadata.name}: {dataset.metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their `@id` values.


In [ ]:
# List all record sets in the dataset
print("Available Record Sets and their @id values:")
record_sets = dataset.record_sets
for rs in record_sets:
    print(f"@id: {rs['@id']}, name: {rs.get('name', 'N/A')}")
    # List all fields (columns) of the record set
    print("  Fields (by @id):")
    for f in rs.get('field', []):
        print(f"    - {f['@id']} (name: {f.get('name','N/A')}, dataType: {f.get('dataType','N/A')})")
    print()

# Store @id values of record sets (for later use)
record_set_ids = [rs['@id'] for rs in record_sets]

## 3. Data Extraction
Load data from specific record sets into DataFrames for analysis, referencing them by their `@id`. Below we extract all records for each record set.


In [ ]:
# Extract data from each record set
dataframes = {}
for record_set_id in record_set_ids:
    print(f"Loading record set: {record_set_id}")
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"\tColumns: {df.columns.tolist()}")
    print(df.head(2), "\n")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, or grouping data. All fields are referenced by their `@id`.

In [ ]:
# Select a record set and field(s) for further analysis
example_record_set_id = record_set_ids[0] if record_set_ids else None
df = dataframes[example_record_set_id]
# List numeric candidate fields
print(f"Numeric fields in {example_record_set_id}:")
for col in df.columns:
    if pd.api.types.is_numeric_dtype(df[col]):
        print(f"- {col}")
# Choose a numeric field (e.g., first numeric field)
numeric_field_id = None
for col in df.columns:
    if pd.api.types.is_numeric_dtype(df[col]):
        numeric_field_id = col
        break

# If a numeric field exists, perform filtering and normalization
if numeric_field_id is not None:
    threshold = df[numeric_field_id].mean() if pd.notnull(df[numeric_field_id].mean()) else 0
    filtered_df = df[df[numeric_field_id] > threshold].copy()
    print(f"Filtered records with {numeric_field_id} > {threshold:.2f}:")
    print(filtered_df.head())

    # Normalize
    filtered_df[f"{numeric_field_id}_normalized"] = (
        (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    )
    print(f"Normalized {numeric_field_id} for filtered records:")
    print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    # Group by a categorical field if available
    group_field_id = None
    for col in df.columns:
        if pd.api.types.is_object_dtype(df[col]) and col != numeric_field_id:
            group_field_id = col
            break

    if group_field_id:
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
        print(f"Grouped average {numeric_field_id} by {group_field_id}:")
        print(grouped_df.head())
else:
    print('No numeric field available for EDA.')

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset using matplotlib and seaborn.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_field_id is not None:
    # Histogram of numeric field
    plt.figure(figsize=(6,4))
    sns.histplot(df[numeric_field_id].dropna(), kde=True, bins=20)
    plt.title(f"Distribution of {numeric_field_id} in {example_record_set_id}")
    plt.xlabel(numeric_field_id)
    plt.show()
    # Boxplot by group if group field available
    if group_field_id:
        plt.figure(figsize=(10,4))
        sns.boxplot(data=filtered_df, x=group_field_id, y=numeric_field_id)
        plt.title(f"{numeric_field_id} by {group_field_id}")
        plt.xticks(rotation=45)
        plt.tight_layout()
        plt.show()

## 6. Conclusion
This notebook demonstrated end-to-end exploration of the FAIR² dataset using the `mlcroissant` library. We loaded the Croissant schema, listed available record sets and fields by their `@id`, loaded the records into pandas DataFrames, and performed basic EDA including filtering, normalization, grouping, and visualization. Further biological and statistical analyses can be easily built on these primitives using this FAIR dataset.